In [ ]:
# --- imports --- #
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut, KFold, cross_val_predict
from sklearn.metrics import r2_score

In [ ]:
# --- User settings Go here --- #

descriptor_path = r"C:\example_data_descripors.csv"     #input your descriptor file here as a CSV
response_variable = "xxxx"                       #input your response variable column name
descriptor_columns = ["yyyy","zzzz"]             #input descriptor column names
cv_method = "Kfold"                                #input cross validation choice Kfold or LOOCV
n_folds = 5                                        #input cross validation fold no.
molecule_ID = "ID"                                #input your run/molecule ID column
# --- Note --- #
#if you are coming from kinetic data obtained using the Analyser 3000, you will need to join the k values to the descriptor CSV.
#if you dont have a descriptor file - check our Joe's Descriptorizer3000, (coming soon, or already out depending on if I forget this is here)

In [ ]:
# --- Read data --- #

k_D = pd.read_csv(descriptor_path)
print(k_D)


In [ ]:
# --- MLR model Fitting --- # 

X = k_D[descriptor_columns]
y = k_D[response_variable]

mlr = LinearRegression()
mlr.fit(X,y)

print(f" R2 score: {mlr.score(X, y):.5f}")
for name, coef in zip(descriptor_columns, mlr.coef_):
    print(f"{name} coefficient: {coef:.5f}")
print(f"Intercept: {mlr.intercept_:5f}")


In [ ]:
# --- Cross Validation --- #

if cv_method.upper() == "LOOCV":
    cv = LeaveOneOut()
elif cv_method.lower() == "kfold":
    cv = KFold(n_splits=n_folds, shuffle=True, random_state=0)
else:
    print("Error probably due to method selected")

y_cv_pred = cross_val_predict(mlr, X, y, cv = cv)
q_squared = r2_score(y, y_cv_pred)

print(f"R2:{mlr.score(X,y):.5f}")
print(f"Q2:{cv_method}):{q_squared:.5f}")


In [ ]:
# --- Prediction vs Measured Plot --- #

fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(1,1,1)
ax.scatter(y, y_cv_pred, color="#70b3f0", edgecolor='k', s=50, alpha=0.8, zorder=3)

raw_labels = k_D[molecule_ID].values

for i in range(len(y)):
    labels = str(raw_labels[i])
    ax.annotate(labels, (np.array(y)[i], y_cv_pred[i]),
        xytext=(5,5), textcoords="offset points", fontsize=10, fontweight="bold", color="#2c3e60")

min_point = min(min(y), min(y_cv_pred))
max_point = max(max(y), max(y_cv_pred))

ax.plot([min_point,max_point], [min_point,max_point], color="black", linestyle="--")

ax.set_xlabel(f"measured {response_variable}", fontsize=12)
ax.set_ylabel(f"predicted {response_variable}", fontsize=12)
ax.set_title(f"Predicted vs Measured, (Q2 = {q_squared:.3f})", fontsize=12)
plt.show()

